In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# SETTINGS
# ============================================================

ROOT = Path(".").resolve()

TRANSITIONS = ["AB", "AC", "BCb"]

FEATURE_SETS = [30, 20, 12]

MODELS_OF_INTEREST = [
    "Logistic Regression",
    "Random Forest",
    "Neural Network",
]

CALIBRATED = "calibrated"
UNCALIBRATED = "uncalibrated"


# ============================================================
# CRITICAL POINT COLUMN FOR EACH TRANSITION
# ============================================================

CRITICAL_COLUMN = {
    "AC": "K0_crit",
    "BCb": "Delta_crit",
    "AB": "Delta_crit",
}


# ============================================================
# HELPERS
# ============================================================

def load_csv(path):

    if not path.exists():
        raise FileNotFoundError(
            f"File not found:\n{path}"
        )

    return pd.read_csv(path)


def filter_data(
    df,
    feature_set,
    model,
    calibration=None,
):

    result = df[
        (df["feature_set"] == feature_set)
        & (df["model"] == model)
    ].copy()

    if (
        calibration is not None
        and "calibration" in result.columns
    ):

        result = result[
            result["calibration"] == calibration
        ].copy()

    return result


# ============================================================
# FIND FILES
# ============================================================

def find_supervised_file(
    transition,
    filename,
):
    """
    Search recursively inside SUPERVISED.
    """

    matches = list(
        (ROOT / "SUPERVISED").rglob(filename)
    )

    if not matches:

        raise FileNotFoundError(
            f"\nCould not find:\n"
            f"    {filename}\n"
            f"for transition: {transition}\n"
            f"inside:\n"
            f"    {ROOT / 'SUPERVISED'}"
        )

    if len(matches) > 1:

        print(
            f"\nWARNING: multiple files found "
            f"for {filename}:"
        )

        for path in matches:
            print("   ", path)

    return matches[0]


# ============================================================
# LOAD MEAN
# ============================================================

def load_curve_mean(transition):

    path = find_supervised_file(
        transition,
        f"{transition}_curve_mean.csv",
    )

    print(
        f"{transition} mean:"
    )
    print(
        f"    {path}"
    )

    return load_csv(path)


# ============================================================
# LOAD CHI
# ============================================================

def load_curve_chi(transition):

    path = find_supervised_file(
        transition,
        f"{transition}_curve_chi.csv",
    )

    print(
        f"{transition} chi:"
    )
    print(
        f"    {path}"
    )

    return load_csv(path)


# ============================================================
# LOAD SUMMARY
# ============================================================

def load_summary(transition):

    path = find_supervised_file(
        transition,
        f"{transition}_summary.csv",
    )

    print(
        f"{transition} summary:"
    )
    print(
        f"    {path}"
    )

    return load_csv(path)


# ============================================================
# GET CRITICAL VALUE
# ============================================================

def get_critical_value(
    summary_df,
    transition,
    feature_set,
    model,
    calibration,
):
    """
    Get the appropriate critical point.

    AC:
        K0_crit

    BCb:
        Delta_crit

    AB:
        defined in CRITICAL_COLUMN above.
    """

    if transition not in CRITICAL_COLUMN:

        raise ValueError(
            f"No critical-point definition "
            f"for transition: {transition}"
        )

    column = CRITICAL_COLUMN[transition]

    if column not in summary_df.columns:

        raise ValueError(
            f"\nColumn '{column}' not found "
            f"in {transition}_summary.csv.\n\n"
            f"Available columns:\n"
            f"{list(summary_df.columns)}"
        )

    result = summary_df[
        (summary_df["feature_set"] == feature_set)
        & (summary_df["model"] == model)
        & (summary_df["calibration"] == calibration)
    ].copy()

    if result.empty:
        return np.nan, np.nan

    values = pd.to_numeric(
        result[column],
        errors="coerce",
    ).dropna()

    if values.empty:
        return np.nan, np.nan

    critical_value = values.iloc[0]

    # --------------------------------------------------------
    # Search for corresponding uncertainty
    # --------------------------------------------------------

    possible_error_columns = [
        f"{column}_err",
        f"{column}_error",
        f"{column}_std",
        f"{column}_se",
    ]

    error = np.nan

    for error_column in possible_error_columns:

        if error_column in result.columns:

            errors = pd.to_numeric(
                result[error_column],
                errors="coerce",
            ).dropna()

            if not errors.empty:

                error = errors.iloc[0]

                break

    return critical_value, error


# ============================================================
# ANALYZE CRITICAL POINT
# ============================================================

def analyze_critical_point(
    summary_df,
    transition,
):

    rows = []

    models = sorted(
        summary_df["model"]
        .dropna()
        .unique()
    )

    for model in models:

        for feature_set in FEATURE_SETS:

            # ------------------------------------------------
            # UNCALIBRATED
            # ------------------------------------------------

            raw, raw_err = get_critical_value(
                summary_df,
                transition,
                feature_set,
                model,
                UNCALIBRATED,
            )

            # ------------------------------------------------
            # CALIBRATED
            # ------------------------------------------------

            cal, cal_err = get_critical_value(
                summary_df,
                transition,
                feature_set,
                model,
                CALIBRATED,
            )

            if (
                not np.isfinite(raw)
                or not np.isfinite(cal)
            ):
                continue

            # ------------------------------------------------
            # DIFFERENCE
            # ------------------------------------------------

            difference = cal - raw

            abs_difference = abs(
                difference
            )

            # ------------------------------------------------
            # COMBINED ERROR
            # ------------------------------------------------

            combined_error = np.nan

            if (
                np.isfinite(raw_err)
                and np.isfinite(cal_err)
            ):

                combined_error = np.sqrt(
                    raw_err**2
                    + cal_err**2
                )

            rows.append({

                "transition":
                    transition,

                "critical_parameter":
                    CRITICAL_COLUMN[transition],

                "model":
                    model,

                "feature_set":
                    feature_set,

                "critical_uncalibrated":
                    raw,

                "critical_calibrated":
                    cal,

                "difference":
                    difference,

                "abs_difference":
                    abs_difference,

                "error_uncalibrated":
                    raw_err,

                "error_calibrated":
                    cal_err,

                "combined_error":
                    combined_error,

                "within_4e-5":
                    abs_difference <= 4e-5,

                "within_statistical_error":
                    (
                        np.isfinite(
                            combined_error
                        )
                        and
                        abs_difference
                        <= combined_error
                    ),
            })

    return pd.DataFrame(rows)


# ============================================================
# GET CHI MAXIMUM
# ============================================================

def get_chi_max(
    chi_df,
    feature_set,
    model,
    calibration,
):

    curve = filter_data(
        chi_df,
        feature_set,
        model,
        calibration,
    )

    if curve.empty:

        return (
            np.nan,
            np.nan,
            np.nan,
        )

    curve = curve.copy()

    curve["chi"] = pd.to_numeric(
        curve["chi"],
        errors="coerce",
    )

    curve = curve.dropna(
        subset=["chi"]
    )

    if curve.empty:

        return (
            np.nan,
            np.nan,
            np.nan,
        )

    idx = curve["chi"].idxmax()

    chi_max = curve.loc[
        idx,
        "chi",
    ]

    # --------------------------------------------------------
    # X AXIS
    # --------------------------------------------------------

    if "K0" in curve.columns:

        x_column = "K0"

    elif "Delta" in curve.columns:

        x_column = "Delta"

    else:

        raise ValueError(
            "Could not find K0 or Delta "
            "in chi dataframe."
        )

    x_max = curve.loc[
        idx,
        x_column,
    ]

    # --------------------------------------------------------
    # CHI ERROR AT MAXIMUM
    # --------------------------------------------------------

    chi_err = np.nan

    if "chi_err" in curve.columns:

        value = pd.to_numeric(
            curve.loc[idx, "chi_err"],
            errors="coerce",
        )

        if np.isfinite(value):

            chi_err = value

    return (
        chi_max,
        x_max,
        chi_err,
    )


# ============================================================
# ANALYZE SUSCEPTIBILITY
# ============================================================

def analyze_susceptibility(
    chi_df,
    transition,
):

    rows = []

    models = sorted(
        chi_df["model"]
        .dropna()
        .unique()
    )

    for model in models:

        for feature_set in FEATURE_SETS:

            # ------------------------------------------------
            # UNCALIBRATED
            # ------------------------------------------------

            raw_chi, raw_x, raw_err = (
                get_chi_max(
                    chi_df,
                    feature_set,
                    model,
                    UNCALIBRATED,
                )
            )

            # ------------------------------------------------
            # CALIBRATED
            # ------------------------------------------------

            cal_chi, cal_x, cal_err = (
                get_chi_max(
                    chi_df,
                    feature_set,
                    model,
                    CALIBRATED,
                )
            )

            if (
                not np.isfinite(raw_chi)
                or not np.isfinite(cal_chi)
            ):
                continue

            # ------------------------------------------------
            # RATIO
            # ------------------------------------------------

            ratio = (
                cal_chi
                / raw_chi
            )

            relative_change = (
                (cal_chi - raw_chi)
                / raw_chi
            )

            # ------------------------------------------------
            # MAXIMUM POSITION
            # ------------------------------------------------

            x_shift = (
                cal_x
                - raw_x
            )

            rows.append({

                "transition":
                    transition,

                "model":
                    model,

                "feature_set":
                    feature_set,

                "chi_max_uncalibrated":
                    raw_chi,

                "chi_max_calibrated":
                    cal_chi,

                "ratio_calibrated_uncalibrated":
                    ratio,

                "relative_change":
                    relative_change,

                "x_max_uncalibrated":
                    raw_x,

                "x_max_calibrated":
                    cal_x,

                "x_shift":
                    x_shift,

                "chi_err_uncalibrated":
                    raw_err,

                "chi_err_calibrated":
                    cal_err,
            })

    return pd.DataFrame(rows)


# ============================================================
# JACKKNIFE ERROR NEAR CRITICAL POINT
# ============================================================

def analyze_jackknife_near_critical(
    chi_df,
    summary_df,
    transition,
    model,
    feature_set,
    window=5,
):
    """
    Compare chi_err for points closest to the
    critical point.

    AC:
        distance from K0_crit

    BCb:
        distance from Delta_crit

    AB:
        distance from the parameter defined above.
    """

    rows = []

    # --------------------------------------------------------
    # UNCALIBRATED CRITICAL POINT
    # --------------------------------------------------------

    raw_crit, raw_crit_err = (
        get_critical_value(
            summary_df,
            transition,
            feature_set,
            model,
            UNCALIBRATED,
        )
    )

    # --------------------------------------------------------
    # CALIBRATED CRITICAL POINT
    # --------------------------------------------------------

    cal_crit, cal_crit_err = (
        get_critical_value(
            summary_df,
            transition,
            feature_set,
            model,
            CALIBRATED,
        )
    )

    # --------------------------------------------------------
    # LOOP OVER CALIBRATION
    # --------------------------------------------------------

    for calibration, critical_point in [

        (
            UNCALIBRATED,
            raw_crit,
        ),

        (
            CALIBRATED,
            cal_crit,
        ),

    ]:

        if not np.isfinite(
            critical_point
        ):
            continue

        curve = filter_data(
            chi_df,
            feature_set,
            model,
            calibration,
        )

        if curve.empty:
            continue

        # ----------------------------------------------------
        # X AXIS
        # ----------------------------------------------------

        if "K0" in curve.columns:

            x_column = "K0"

        elif "Delta" in curve.columns:

            x_column = "Delta"

        else:

            raise ValueError(
                "Could not find K0 or Delta "
                "in chi dataframe."
            )

        # ----------------------------------------------------
        # DISTANCE FROM CRITICAL POINT
        # ----------------------------------------------------

        curve = curve.copy()

        curve["distance_from_critical"] = (
            abs(
                curve[x_column]
                - critical_point
            )
        )

        curve = curve.sort_values(
            "distance_from_critical"
        ).head(window)

        # ----------------------------------------------------
        # STORE
        # ----------------------------------------------------

        for _, row in curve.iterrows():

            rows.append({

                "transition":
                    transition,

                "model":
                    model,

                "feature_set":
                    feature_set,

                "calibration":
                    calibration,

                "critical_parameter":
                    CRITICAL_COLUMN[transition],

                "critical_value":
                    critical_point,

                "x":
                    row[x_column],

                "distance_from_critical":
                    row[
                        "distance_from_critical"
                    ],

                "chi_err":
                    row["chi_err"],
            })

    return pd.DataFrame(rows)


# ============================================================
# ANALYZE PROBABILITY EXTREMES
# ============================================================

def analyze_probability_extremes(
    mean_df,
    transition,
):

    rows = []

    models = sorted(
        mean_df["model"]
        .dropna()
        .unique()
    )

    for model in models:

        for feature_set in FEATURE_SETS:

            for calibration in [

                UNCALIBRATED,
                CALIBRATED,

            ]:

                curve = filter_data(
                    mean_df,
                    feature_set,
                    model,
                    calibration,
                )

                if curve.empty:
                    continue

                curve = curve.copy()

                curve["mean"] = pd.to_numeric(
                    curve["mean"],
                    errors="coerce",
                )

                curve = curve.dropna(
                    subset=["mean"]
                )

                if curve.empty:
                    continue

                # ------------------------------------------------
                # X AXIS
                # ------------------------------------------------

                if "K0" in curve.columns:

                    x_column = "K0"

                elif "Delta" in curve.columns:

                    x_column = "Delta"

                else:

                    raise ValueError(
                        "Could not find K0 or Delta "
                        "in mean dataframe."
                    )

                # ------------------------------------------------
                # MIN / MAX
                # ------------------------------------------------

                min_idx = curve[
                    "mean"
                ].idxmin()

                max_idx = curve[
                    "mean"
                ].idxmax()

                min_value = curve.loc[
                    min_idx,
                    "mean",
                ]

                max_value = curve.loc[
                    max_idx,
                    "mean",
                ]

                rows.append({

                    "transition":
                        transition,

                    "model":
                        model,

                    "feature_set":
                        feature_set,

                    "calibration":
                        calibration,

                    "P_min":
                        min_value,

                    "P_max":
                        max_value,

                    "x_at_P_min":
                        curve.loc[
                            min_idx,
                            x_column,
                        ],

                    "x_at_P_max":
                        curve.loc[
                            max_idx,
                            x_column,
                        ],

                    "distance_from_0":
                        min_value,

                    "distance_from_1":
                        1 - max_value,
                })

    return pd.DataFrame(rows)


# ============================================================
# PRINT CRITICAL POINT RESULTS
# ============================================================

def print_critical_results(df):

    print("\n")
    print("=" * 110)
    print(
        "1. CRITICAL POINT — "
        "CALIBRATED VS UNCALIBRATED"
    )
    print("=" * 110)

    if df.empty:

        print("No results.")

        return

    columns = [

        "transition",

        "critical_parameter",

        "model",

        "feature_set",

        "critical_uncalibrated",

        "critical_calibrated",

        "abs_difference",

        "combined_error",

        "within_4e-5",

        "within_statistical_error",
    ]

    print(
        df[columns].to_string(
            index=False,
            float_format=lambda x:
                f"{x:.8f}",
        )
    )

    # --------------------------------------------------------
    # MAXIMUM DIFFERENCE
    # --------------------------------------------------------

    print("\n")

    maximum = df[
        "abs_difference"
    ].max()

    print(
        "Maximum absolute difference:"
    )

    print(
        f"{maximum:.10f}"
    )

    print(
        "\nAll differences <= 4e-5:"
    )

    print(
        df["within_4e-5"].all()
    )

    # --------------------------------------------------------
    # STATISTICAL ERROR
    # --------------------------------------------------------

    valid = df[
        df["combined_error"].notna()
    ]

    if not valid.empty:

        print(
            "\nDifferences within "
            "combined statistical uncertainty:"
        )

        print(
            valid[
                "within_statistical_error"
            ].all()
        )


# ============================================================
# PRINT SUSCEPTIBILITY RESULTS
# ============================================================

def print_susceptibility_results(df):

    print("\n")
    print("=" * 110)
    print(
        "2. SUSCEPTIBILITY AMPLITUDE "
        "AND MAXIMUM POSITION"
    )
    print("=" * 110)

    if df.empty:

        print("No results.")

        return

    columns = [

        "transition",

        "model",

        "feature_set",

        "chi_max_uncalibrated",

        "chi_max_calibrated",

        "ratio_calibrated_uncalibrated",

        "relative_change",

        "x_max_uncalibrated",

        "x_max_calibrated",

        "x_shift",

    ]

    print(
        df[columns].to_string(
            index=False,
            float_format=lambda x:
                f"{x:.8g}",
        )
    )


# ============================================================
# PRINT PROBABILITY RESULTS
# ============================================================

def print_probability_results(df):

    print("\n")
    print("=" * 110)
    print(
        "3. PROBABILITY EXTREMES"
    )
    print("=" * 110)

    if df.empty:

        print("No results.")

        return

    print(
        df.to_string(
            index=False,
            float_format=lambda x:
                f"{x:.10g}",
        )
    )


# ============================================================
# PRINT JACKKNIFE RESULTS
# ============================================================

def print_jackknife_results(df):

    print("\n")
    print("=" * 110)
    print(
        "4. JACKKNIFE ERROR NEAR CRITICAL POINT"
    )
    print("=" * 110)

    if df.empty:

        print("No results.")

        return

    print(
        df.to_string(
            index=False,
            float_format=lambda x:
                f"{x:.8g}",
        )
    )


# ============================================================
# MAIN
# ============================================================

all_critical_results = []

all_chi_results = []

all_probability_results = []

all_jackknife_results = []


for transition in TRANSITIONS:

    print("\n\n")

    print(
        "#" * 110
    )

    print(
        f"# TRANSITION: {transition}"
    )

    print(
        "#" * 110
    )

    # ========================================================
    # LOAD
    # ========================================================

    mean_df = load_curve_mean(
        transition
    )

    chi_df = load_curve_chi(
        transition
    )

    summary_df = load_summary(
        transition
    )

    # ========================================================
    # PRINT AVAILABLE COLUMNS
    # ========================================================

    print("\nSummary columns:")

    print(
        list(summary_df.columns)
    )

    # ========================================================
    # CRITICAL POINT
    # ========================================================

    critical_results = (
        analyze_critical_point(
            summary_df,
            transition,
        )
    )

    all_critical_results.append(
        critical_results
    )

    print_critical_results(
        critical_results
    )

    # ========================================================
    # SUSCEPTIBILITY
    # ========================================================

    chi_results = (
        analyze_susceptibility(
            chi_df,
            transition,
        )
    )

    all_chi_results.append(
        chi_results
    )

    print_susceptibility_results(
        chi_results
    )

    # ========================================================
    # PROBABILITY EXTREMES
    # ========================================================

    probability_results = (
        analyze_probability_extremes(
            mean_df,
            transition,
        )
    )

    all_probability_results.append(
        probability_results
    )

    print_probability_results(
        probability_results
    )

    # ========================================================
    # JACKKNIFE
    # ========================================================

    for model in MODELS_OF_INTEREST:

        for feature_set in FEATURE_SETS:

            result = (
                analyze_jackknife_near_critical(
                    chi_df=chi_df,
                    summary_df=summary_df,
                    transition=transition,
                    model=model,
                    feature_set=feature_set,
                    window=5,
                )
            )

            if not result.empty:

                all_jackknife_results.append(
                    result
                )


# ============================================================
# COMBINE
# ============================================================

critical_results_all = pd.concat(
    all_critical_results,
    ignore_index=True,
)

chi_results_all = pd.concat(
    all_chi_results,
    ignore_index=True,
)

probability_results_all = pd.concat(
    all_probability_results,
    ignore_index=True,
)

if all_jackknife_results:

    jackknife_results_all = pd.concat(
        all_jackknife_results,
        ignore_index=True,
    )

else:

    jackknife_results_all = pd.DataFrame()


# ============================================================
# OUTPUT DIRECTORY
# ============================================================

output_dir = (
    ROOT
    / "calibration_verification"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# SAVE RESULTS
# ============================================================

critical_results_all.to_csv(
    output_dir
    / "critical_point_comparison.csv",
    index=False,
)

chi_results_all.to_csv(
    output_dir
    / "susceptibility_comparison.csv",
    index=False,
)

probability_results_all.to_csv(
    output_dir
    / "probability_extremes.csv",
    index=False,
)

if not jackknife_results_all.empty:

    jackknife_results_all.to_csv(
        output_dir
        / "jackknife_near_critical.csv",
        index=False,
    )


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n\n")

print(
    "=" * 110
)

print(
    "FILES SAVED"
)

print(
    "=" * 110
)

print(
    output_dir
    / "critical_point_comparison.csv"
)

print(
    output_dir
    / "susceptibility_comparison.csv"
)

print(
    output_dir
    / "probability_extremes.csv"
)

if not jackknife_results_all.empty:

    print(
        output_dir
        / "jackknife_near_critical.csv"
    )




##############################################################################################################
# TRANSITION: AB
##############################################################################################################
AB mean:
    /home/mariuszoslaw/uni/masters/Results/SUPERVISED/AB/AB_curve_mean.csv
AB chi:
    /home/mariuszoslaw/uni/masters/Results/SUPERVISED/AB/AB_curve_chi.csv
AB summary:
    /home/mariuszoslaw/uni/masters/Results/SUPERVISED/AB/AB_summary.csv

Summary columns:
['feature_set', 'n_features', 'model', 'calibration', 'train_acc', 'Delta_crit', 'Delta_crit_err', 'Delta_close', 'N_Delta', 'runtime_model', 'runtime_jackknife_mean', 'runtime_jackknife_chi', 'summary_mean_time', 'summary_chi_time', 'runtime_delta_crit', 'runtime_total']


1. CRITICAL POINT — CALIBRATED VS UNCALIBRATED
transition critical_parameter                           model  feature_set  critical_uncalibrated  critical_calibrated  abs_difference  combined_error  within_4e-5  wit

/tmp/ipykernel_25850/3637556661.py:559: RuntimeWarning: divide by zero encountered in scalar divide
  cal_chi
/tmp/ipykernel_25850/3637556661.py:564: RuntimeWarning: divide by zero encountered in scalar divide
  (cal_chi - raw_chi)
/tmp/ipykernel_25850/3637556661.py:559: RuntimeWarning: divide by zero encountered in scalar divide
  cal_chi
/tmp/ipykernel_25850/3637556661.py:564: RuntimeWarning: divide by zero encountered in scalar divide
  (cal_chi - raw_chi)
/tmp/ipykernel_25850/3637556661.py:559: RuntimeWarning: divide by zero encountered in scalar divide
  cal_chi
/tmp/ipykernel_25850/3637556661.py:564: RuntimeWarning: divide by zero encountered in scalar divide
  (cal_chi - raw_chi)
/tmp/ipykernel_25850/3637556661.py:559: RuntimeWarning: divide by zero encountered in scalar divide
  cal_chi
/tmp/ipykernel_25850/3637556661.py:564: RuntimeWarning: divide by zero encountered in scalar divide
  (cal_chi - raw_chi)
/tmp/ipykernel_25850/3637556661.py:559: RuntimeWarning: divide by zero e



3. PROBABILITY EXTREMES
transition                           model  feature_set  calibration           P_min        P_max  x_at_P_min  x_at_P_max  distance_from_0  distance_from_1
        AB                   Decision Tree           30 uncalibrated               0            1      -0.114      -0.128                0                0
        AB                   Decision Tree           30   calibrated  0.000831808446 0.9997501249      -0.114      -0.124   0.000831808446  0.0002498750637
        AB                   Decision Tree           20 uncalibrated               0            1      -0.114      -0.128                0                0
        AB                   Decision Tree           20   calibrated  0.000831808446 0.9997501249      -0.114      -0.124   0.000831808446  0.0002498750637
        AB                   Decision Tree           12 uncalibrated               0            1      -0.114      -0.128                0                0
        AB                   Decision 